# Chapter 32: Multiview Geometry

<a href="../lite/lab/index.html?path=ch32_multiview_geometry.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

Take two photos of the same scene from different positions. Each photo alone is flat, with
no depth. But together, they contain enough information to reconstruct the 3D world.
The mathematical key is the **epipolar constraint**: a point in image 1 constrains where
its match can appear in image 2 to a single line.

## 32.1 Epipolar Geometry

The **essential matrix** $E$ relates corresponding points in two views:

$$\mathbf{p}_2^T E \mathbf{p}_1 = 0$$

where $\mathbf{p}_1, \mathbf{p}_2$ are normalized image coordinates.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
n_points = 20
baseline = 1.0          # camera separation (meters)   (try 0.5, 1.0, 2.0)
depth_range = (3, 8)    # depth range of 3D points
fx, fy = 500, 500
cx, cy = 320, 240
# ──────────────────────────────────────────────────────────────────────────────

K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])

# 3D points
points_3d = np.column_stack([
    np.random.uniform(-3, 3, n_points),
    np.random.uniform(-2, 2, n_points),
    np.random.uniform(*depth_range, n_points)
])

# Camera 1 at origin, Camera 2 translated along X
R1 = np.eye(3); t1 = np.zeros(3)
R2 = np.eye(3); t2 = np.array([baseline, 0, 0])

# Project to both cameras
def project(K, R, t, pts):
    pts_cam = (R @ (pts - t).T).T
    proj = (K @ pts_cam.T).T
    return proj[:, :2] / proj[:, 2:3], pts_cam[:, 2]

pix1, d1 = project(K, R1, t1, points_3d)
pix2, d2 = project(K, R2, t2, points_3d)

# Essential matrix
tx = np.array([[0, -t2[2], t2[1]], [t2[2], 0, -t2[0]], [-t2[1], t2[0], 0]])
E = tx @ R2  # E = [t]x @ R

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(pix1[:, 0], pix1[:, 1], c='steelblue', s=40, zorder=5)
for i in range(n_points):
    ax.annotate(str(i), xy=pix1[i], fontsize=7, color='steelblue')
ax.set_xlim(0, 640); ax.set_ylim(480, 0); ax.set_aspect('equal')
ax.set_title("Camera 1", fontsize=13)

ax = axes[1]
ax.scatter(pix2[:, 0], pix2[:, 1], c='tomato', s=40, zorder=5)
for i in range(n_points):
    ax.annotate(str(i), xy=pix2[i], fontsize=7, color='tomato')
ax.set_xlim(0, 640); ax.set_ylim(480, 0); ax.set_aspect('equal')
ax.set_title(f"Camera 2 (baseline = {baseline}m)", fontsize=13)

plt.tight_layout()
plt.show()

## 32.2 Essential Matrix

The essential matrix encodes the relative pose (R, t) between two cameras.
It has exactly 5 degrees of freedom (3 rotation + 2 translation direction, scale is unknown).

## 32.3 Triangulation

Given corresponding points in two images and the camera poses, we can **triangulate**
the 3D position by finding the intersection of the two back-projected rays.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
pixel_noise = 1.0      # pixel noise (try 0, 1, 5)
# ──────────────────────────────────────────────────────────────────────────────

# Add noise to pixel observations
np.random.seed(42)
pix1_noisy = pix1 + np.random.normal(0, pixel_noise, pix1.shape)
pix2_noisy = pix2 + np.random.normal(0, pixel_noise, pix2.shape)

# Triangulate using DLT
K_inv = np.linalg.inv(K)
triangulated = []
for i in range(n_points):
    # Normalized coordinates
    p1 = K_inv @ np.append(pix1_noisy[i], 1)
    p2 = K_inv @ np.append(pix2_noisy[i], 1)
    # Build A matrix for DLT
    A = np.zeros((4, 4))
    P1 = K @ np.hstack([R1, -R1 @ t1.reshape(3,1)])
    P2 = K @ np.hstack([R2, -R2 @ t2.reshape(3,1)])
    u1, v1 = pix1_noisy[i]
    u2, v2 = pix2_noisy[i]
    A[0] = u1 * P1[2] - P1[0]
    A[1] = v1 * P1[2] - P1[1]
    A[2] = u2 * P2[2] - P2[0]
    A[3] = v2 * P2[2] - P2[1]
    _, _, Vt = np.linalg.svd(A)
    X = Vt[-1]; X = X[:3] / X[3]
    triangulated.append(X)
triangulated = np.array(triangulated)

errors = np.linalg.norm(triangulated - points_3d, axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(points_3d[:, 0], points_3d[:, 2], c='steelblue', s=60, label='true 3D')
ax.scatter(triangulated[:, 0], triangulated[:, 2], c='tomato', s=40, marker='x', label='triangulated')
for i in range(n_points):
    ax.plot([points_3d[i,0], triangulated[i,0]], [points_3d[i,2], triangulated[i,2]], 'gray', alpha=0.3)
ax.plot(0, 0, 'k^', ms=12, label='Camera 1')
ax.plot(baseline, 0, 'ks', ms=12, label='Camera 2')
ax.set_xlabel("X (m)"); ax.set_ylabel("Z (m)")
ax.set_title(f"Triangulation (pixel noise σ = {pixel_noise}px, mean error = {errors.mean():.3f}m)", fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

## 32.4 Relative Pose

The relative pose between two cameras can be recovered from the essential matrix
using SVD decomposition. There are 4 possible solutions; the correct one is identified
by checking which places all triangulated points in front of both cameras.

**Key observations:**
- Two views are the minimum for 3D reconstruction.
- Triangulation accuracy depends on the **baseline to depth ratio**. Larger baseline = better depth.
- Pixel noise has a larger effect on far away points than on nearby ones.
- The essential matrix is the foundation of visual SLAM initialization.

---

## Exercises

### Exercise 32.1
Vary the baseline from 0.1m to 5m. Plot the mean triangulation error vs baseline.
What is the optimal baseline for points at depth 5m?

### Exercise 32.2 (challenge)
Estimate the essential matrix from 8 or more point correspondences using the 8-point
algorithm. Decompose into R, t and compare with the true relative pose.

In [ ]:
# Your code here